In [9]:
from sumo_rl.environment.env import SumoEnvironment

In [10]:
sumo_env = SumoEnvironment(
        net_file="single-intersection.net.xml",
        route_file="single-intersection-vhvh.rou.xml",
        use_gui=False,
        num_seconds=1000,
        yellow_time=3,
        min_green=5
    )

In [ ]:
# traffic signal id
print(sumo_env.ts_ids)

In [ ]:
### obs = [phase_one_hot, min_green, lane_1_density,...,lane_n_density, lane_1_queue,...,lane_n_queue]
### phase_one_hot is a one-hot encoded vector indicating the current active green phase
### min_green is a binary variable indicating whether min_green seconds have already passed in the current phase
### lane_i_density is the number of vehicles in incoming lane i dividided by the total capacity of the lane
### lane_i_queueis the number of queued (speed below 0.1 m/s) vehicles in incoming lane i divided by the total capacity of the lane

obs = sumo_env.reset()
print(obs)  

In [ ]:
### The action space is discrete. Every 'delta_time' seconds, each traffic signal agent can choose the next green phase configuration.
### Important: every time a phase change occurs, the next phase is preeceded by a yellow phase lasting yellow_time seconds.

print(sumo_env.action_space)

In [14]:
import numpy as np
from gymnasium import spaces, Env
import traci

class SumoEmissionWrapper(Env):
    def __init__(self, env):
        super().__init__()
        self.env = env
        example_obs = self.env.reset()
        flat_obs = self._flatten_obs(example_obs)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=flat_obs.shape, dtype=np.float32)
        self.action_space = env.action_space
        self.previous_waiting_time = 0
        self.previous_vehicles = set()

    def _flatten_obs(self, obs):
        return np.concatenate([np.array(v).flatten() for v in obs.values()]).astype(np.float32)

    def _get_emissions(self):
        vehicle_list = traci.vehicle.getIDList()
        total_co2 = 0.0
        for vehicle_id in vehicle_list:
            co2 = traci.vehicle.getCO2Emission(vehicle_id)
            total_co2 += co2
        return total_co2

    def _get_traffic_metrics(self):
        # Get current vehicles
        current_vehicles = set(traci.vehicle.getIDList())
        
        # Calculate completed trips (vehicles that were present before but are now gone)
        completed_trips = len(self.previous_vehicles - current_vehicles)
        
        # Calculate total waiting time
        total_waiting_time = 0
        total_queue_length = 0
        
        for vehicle_id in current_vehicles:
            # Waiting time is the time (in seconds) spent with speed below 0.1m/s
            waiting_time = traci.vehicle.getAccumulatedWaitingTime(vehicle_id)
            total_waiting_time += waiting_time
            
            # Consider a vehicle queued if its speed is below 0.1 m/s
            if traci.vehicle.getSpeed(vehicle_id) < 0.1:
                total_queue_length += 1
        
        # Update previous vehicles set
        self.previous_vehicles = current_vehicles
        
        return total_waiting_time, total_queue_length, completed_trips

    def reset(self, **kwargs):
        obs = self.env.reset()
        self.previous_waiting_time = 0
        self.previous_vehicles = set(traci.vehicle.getIDList())
        return self._flatten_obs(obs), {}

    def step(self, action):
        ts_id = self.env.ts_ids[0]
        dict_action = {ts_id: int(action)}

        obs, _, done, info = self.env.step(dict_action)
        flat_obs = self._flatten_obs(obs)

        # Get all metrics
        emissions = self._get_emissions()
        waiting_time, queue_length, completed_trips = self._get_traffic_metrics()
        
        # Calculate reward components with more conservative scaling
        emission_penalty = -emissions / 1000000.0  # Further reduce emission impact
        waiting_time_penalty = -waiting_time / 10000.0  # Further reduce waiting time impact
        queue_penalty = -queue_length / 10.0  # Reduce queue impact
        completion_reward = completed_trips * 20.0  # More moderate completion reward
        
        # Normalize the components to similar scales
        max_emission = 200000000  # Based on baseline ~160M
        max_waiting = 20000000    # Based on baseline ~17M
        max_queue = 100           # Reasonable maximum queue length
        max_completion = 50       # Reasonable completions per step
        
        # Normalize each component to roughly [-1, 1] range
        norm_emission = emission_penalty * (1000000 / max_emission)
        norm_waiting = waiting_time_penalty * (10000 / max_waiting)
        norm_queue = queue_penalty * (10 / max_queue)
        norm_completion = completion_reward * (20 / max_completion)
        
        # Combine normalized rewards with adjusted weights
        reward = (0.05 * norm_emission +       # 20% weight for emissions
                 0.3 * norm_waiting +         # 30% weight for waiting time
                 0.35 * norm_queue +           # 20% weight for queue length
                 0.3 * norm_completion)       # 30% weight for completed trips
        
        # Final scaling to keep rewards manageable
        reward = reward * 0.1

        if isinstance(done, dict):
            done_flag = any(done.values())
        else:
            done_flag = done

        return flat_obs, reward, done_flag, False, info

    def render(self):
        return self.env.render()

    def close(self):
        return self.env.close()


In [15]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

def make_env():
    wrapped_env = SumoEmissionWrapper(sumo_env)
    monitored_env = Monitor(wrapped_env)
    return monitored_env

env = DummyVecEnv([make_env])

In [ ]:
from sb3_contrib import QRDQN

model = QRDQN(
    policy="MlpPolicy",
    env=env,
    learning_rate=1e-4,  # Reduced to improve stability
    buffer_size=200000,  # Increased for better experience replay
    learning_starts=5000,  # Increased to gather more initial experiences
    batch_size=128,  # Increased for more stable gradients
    tau=0.1,  # Reduced for more stable target network updates
    gamma=0.95,  # Slightly reduced to focus more on immediate rewards
    train_freq=8,  # Increased to reduce update frequency
    gradient_steps=2,  # Increased to learn more from each batch
    target_update_interval=2000,  # Increased for more stable learning
    exploration_fraction=0.3,  # Increased exploration period
    exploration_final_eps=0.05,  # Increased minimum exploration
    verbose=1
)

# Increase training time
model.learn(total_timesteps=50000)  # Increased from 10000

# model saving
model.save("rainbowDQN_sumo_model_emmision")

# model testing
test_env = DummyVecEnv([lambda: Monitor(make_env())])
obs = test_env.reset()
done = False
total_reward = 0

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    total_reward += reward[0]

print(f"Total reward (test): {total_reward}")

In [20]:
new_env = SumoEnvironment(
    net_file="single-intersection.net.xml",
    route_file="single-intersection-vhvh.rou.xml",
    use_gui=True,  
    num_seconds=1000,
    yellow_time=3,
    min_green=5
)

wrapped_env = SumoEmissionWrapper(new_env)

obs, _ = wrapped_env.reset()
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _, info = wrapped_env.step(action)
    wrapped_env.render()  